In [2]:
import torch
import timm
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
%load_ext autoreload
 
%autoreload 2

model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True)
# model.load_state_dict(torch.load("swin/Swin-Transformer/checkpoints/swin_tiny_patch4_window7_224.pth"))
model.eval()


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.eval()
# Validation function for PyTorch model with progress bar
def validate_pytorch(model, dataloader, device):
    model.eval()
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating PyTorch Model", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred_top1 = outputs.topk(1, dim=1)
            _, pred_top5 = outputs.topk(5, dim=1)

            top1_correct += (pred_top1.squeeze() == labels).sum().item()
            top5_correct += sum([labels[i] in pred_top5[i] for i in range(len(labels))])
            total += labels.size(0)

    top1_acc = 100 * top1_correct / total
    top5_acc = 100 * top5_correct / total
    return top1_acc, top5_acc

# Define ImageNet validation dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder("/media/bmw/datasets/imagenet-1k/val", transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Validate model
# top1_acc, top5_acc = validate_pytorch(model, dataloader, device)
# print(f"Top-1 Accuracy: {top1_acc:.2f}%")
# print(f"Top-5 Accuracy: {top5_acc:.2f}%")


/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
from aimet_torch.model_preparer import prepare_model
from aimet_torch.model_validator.model_validator import ModelValidator
 
print("\nPreparing and validating the model for quantization...")
prepared_model = prepare_model(model)

invalid_layers = ModelValidator.validate_model(prepared_model, model_input=torch.randn(1, 3, 224, 224).to(device))

if not (invalid_layers):
    print("\n❌ Model contains unsupported layers for AIMET quantization:")
else:
    print("\n✅ Model is valid for AIMET quantization!")



2025-03-02 23:04:43,072 - root - INFO - AIMET

Preparing and validating the model for quantization...
2025-03-02 23:04:44,442 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.module_floordiv} 
2025-03-02 23:04:44,443 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.module_floordiv_1} 
2025-03-02 23:04:44,444 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_floordiv_2} 
2025-03-02 23:04:44,444 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_mul} 
2025-03-02 23:04:44,444 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_matmul} 
2025-03-02 23:04:44,445 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_add} 
2025-03-02 23:04:44,445 - ModelPreparer - INFO - Functional         : Adding

/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,


2025-03-02 23:04:44,485 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.module_floordiv_40} 
2025-03-02 23:04:44,485 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.module_floordiv_41} 
2025-03-02 23:04:44,485 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.attn.module_floordiv_42} 
2025-03-02 23:04:44,486 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.attn.module_mul_11} 
2025-03-02 23:04:44,486 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.attn.module_matmul_18} 
2025-03-02 23:04:44,486 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.attn.module_add_31} 
2025-03-02 23:04:44,487 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.2.blocks.5.attn.module_floordiv_43} 
2025-03-02 23:04:44,4

In [5]:
print(type(prepared_model))

<class 'torch.fx.graph_module.GraphModule.__new__.<locals>.GraphModuleImpl'>


In [4]:

from aimet_torch.meta.connectedgraph import ConnectedGraph
 
operations = [
            "floor_divide", "remainder", "pad", "dropout", "Add", "linear", "pow",
            "Mul", "matmul", "softmax", "rsub", "roll", "fill", "masked_fill",
            "new_zeros", "sub", "Concat"
        ]

ConnectedGraph.math_invariant_types.update(operations)

In [5]:
invalid_layers = ModelValidator.validate_model(prepared_model, model_input=torch.randn(1, 3, 224, 224).to(device))

if not (invalid_layers):
    print("\n❌ Model contains unsupported layers for AIMET quantization:")
else:
    print("\n✅ Model is valid for AIMET quantization!")


2025-03-02 23:05:11,806 - Utils - INFO - Running validator check <function validate_for_reused_modules at 0x7f4d59c79870>
2025-03-02 23:05:11,822 - Utils - INFO - Running validator check <function validate_for_missing_modules at 0x7f4d59c79b40>
2025-03-02 23:05:21,811 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-03-02 23:05:21,837 - Utils - INFO - All validation checks passed.

✅ Model is valid for AIMET quantization!


In [6]:
dummy_input = torch.rand(1, 3, 224, 224).to(device)

In [7]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset
from torch.utils.data import Dataset

In [9]:
TRAIN_DIR="/media/bmw/datasets/imagenet-1k/train"
VAL_DIR="/media/bmw/datasets/imagenet-1k/val"

def load_labbeled_data(num_of_classes:int =50, images_per_class:int=10,batch_size:int =128,isTrain:bool =False) -> DataLoader :
    
    tranform_data = transforms.Compose([
        transforms.Resize(249, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[123.675/255, 116.28/255, 103.53/255], std=[58.395/255, 57.12/255, 57.375/255])
    ])

    if isTrain:
        DATA_DIR=TRAIN_DIR
    else:
        DATA_DIR=VAL_DIR
        
    dataset = ImageFolder(
                    root=DATA_DIR,
                    transform=tranform_data
    )


    all_classes=dataset.classes

    # Filter dataset to include only a subset of classes
    selected_classes = all_classes[:num_of_classes]

    # Create a mapping of selected class names to their indices
    selected_class_indices = [dataset.class_to_idx[class_name] for class_name in selected_classes]

    # Get indices of samples belonging to selected classes
    selected_indices = []
    class_counts = {idx: 0 for idx in selected_class_indices}

    for idx, (_, label) in enumerate(dataset.samples):
        if label in selected_class_indices and class_counts[label] < images_per_class:
            selected_indices.append(idx)
            class_counts[label] += 1

    # Create a subset dataset
    subset_dataset = Subset(dataset, selected_indices)

    dataloader = DataLoader (
    subset_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory= True
    )
    
    print(f"Dataset size {len(subset_dataset)} DataLoader {len(dataloader)}")
    
    return subset_dataset,dataloader

    

def load_unlabelled_data(num_of_classes:int =50, images_per_class:int=10,batch_size:int =128) -> DataLoader :
    
    
    dataset,dataloader =load_labbeled_data(num_of_classes,images_per_class,batch_size)

    class UnlabelledDataset(ImageFolder):
        def __init__(self, dataset):
            self._dataset = dataset

        def __len__(self):
            return len(self._dataset)

        def __getitem__(self, index):
            images, _ = self._dataset[index]
            return images
        
    unlabelled_dataset=UnlabelledDataset(dataset)
    
    unlabelled_dataloader= DataLoader (
    unlabelled_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory= True
    )
    
    return unlabelled_dataset,unlabelled_dataloader





In [10]:
from typing import Optional
from aimet_torch.utils import in_eval_mode, get_device

from tqdm import tqdm

def eval_callback(model: torch.nn.Module, num_samples: Optional[int] = None) -> float:
    dataset, dataloader = load_labbeled_data(10, 10, 32)
    device = get_device(model)

    correct_top1 = 0
    correct_top5 = 0
    total = 0  # Track total samples
    
    with in_eval_mode(model), torch.no_grad():
        for image, label in tqdm(dataloader):
            image = image.to(device)
            label = label.to(device)

            logits = model(image)
            topk = logits.topk(k=5).indices  # Get top 5 predictions

            correct_top1 += (topk[:, 0] == label).sum().item()  # Top-1 matches
            correct_top5 += (topk == label.view(-1, 1)).sum().item()  # Top-5 matches
            total += label.size(0)

    top1_acc = correct_top1 / total
    top5_acc = correct_top5 / total

    print(f"Top-1 Accuracy: {top1_acc:.4f}")
    print(f"Top-5 Accuracy: {top5_acc:.4f}")

    return top1_acc # Return both accuracies


In [11]:
eval_callback(prepared_model)

# dataset, dataloader = load_labbeled_data(1000, 100, 32)
# Dataset size 50000 DataLoader 1563
# 100%|██████████| 1563/1563 [09:21<00:00,  2.78it/s]
# Top-1 Accuracy: 0.8140
# Top-5 Accuracy: 0.9551

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:02<00:00,  2.00it/s]

Top-1 Accuracy: 0.9300
Top-5 Accuracy: 0.9900


0.93

In [12]:

dataset,dataloader =load_unlabelled_data(20,20,32)

Dataset size 400 DataLoader 13


AutoQuant

In [ ]:
from aimet_torch.auto_quant import AutoQuant
auto_quant = AutoQuant(prepared_model, dummy_input=torch.randn(1, 3, 224, 224).to(device),data_loader=dataloader,eval_callback=eval_callback, results_dir="/media/bmw/harisharajan_r_r/swin/artifacts/tmpr")

: 

In [ ]:
sim, initial_accuracy =  auto_quant.run_inference()
print(f"- Quantized Accuracy : {initial_accuracy}")

2025-03-02 22:16:23,682 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_mul} 


2025-03-02 22:16:23,684 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_matmul} 


2025-03-02 22:16:23,686 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_add} 


2025-03-02 22:16:23,687 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_matmul_1} 


2025-03-02 22:16:23,689 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_add_1} 


...

2025-03-02 22:18:58,121 - AutoQuant - INFO - The results of Batchnorm Folding is saved in /media/bmw/harisharajan_r_r/swin/artifacts/tmpr/.trace/batchnorm_folding.pth and /media/bmw/harisharajan_r_r/swin/artifacts/tmpr/.trace/batchnorm_folding.encodings.


2025-03-02 22:18:58,126 - AutoQuant - INFO - Session finished: Batchnorm Folding. (eval score: 0.830000)


\ Batchnorm Folding


2025-03-02 22:19:13,091 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-03-02 22:19:13,451 - Quant - INFO - No config file provided, defaulting to config file at /home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/aimet_common/quantsim_config/default_config.json
2025-03-02 22:19:13,473 - Quant - INFO - Unsupported op type Squeeze
2025-03-02 22:19:13,474 - Quant - INFO - Unsupported op type Mean
2025-03-02 22:19:13,495 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default
2025-03-02 22:19:13,499 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-03-02 22:19:13,500 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-03-02 22:19:13,500 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
202

  8%|▊         | 1/13 [00:01<00:17,  1.48s/it]

Applying AdaRound 

In [ ]:
from aimet_torch.adaround.adaround_weight import AdaroundParameters


adaround_params = AdaroundParameters(dataloader, num_batches=len(dataloader), default_num_iterations=2000)
auto_quant.set_adaround_params(adaround_params)

/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


2025-03-02 22:15:23,917 - root - INFO - AIMET


/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,


NameError: name 'dataloader' is not defined

In [ ]:
model, optimized_accuracy, encoding_path , pareto_front= auto_quant.optimize(allowed_accuracy_drop=0.01)
print(f"- Quantized Accuracy (after optimization):  {optimized_accuracy}")

NameError: name 'auto_quant' is not defined

In [1]:
device

NameError: name 'device' is not defined

In [ ]:
sim.model.to(device)

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [ ]:
print(next(sim.model.parameters()).device)


cuda:0


In [ ]:
sim.export(
    "/media/bmw/harisharajan_r_r/swin/artifacts/autoquant", filename_prefix="auto_quant_model", dummy_input=torch.randn(1, 3, 224, 224).to('cpu'), )

2025-02-28 16:54:49,814 - Utils - INFO - successfully created onnx model with 286/1981 node names updated
2025-02-28 16:54:50,985 - Quant - INFO - Layers excluded from quantization: []
2025-02-28 16:54:51,042 - Quant - WARNING - Quantsim export will stop exporting encodings for saving and loading in a future AIMET release.
To export encodings for saving and loading, use QuantizationSimModel's save_encodings_to_json() utility instead.


Quantization Aware Training

In [13]:
print(prepared_model)

GraphModule(
  (patch_embed): Module(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Module(
    (0): Module(
      (blocks): Module(
        (0): Module(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): Module(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (softmax): Softmax(dim=-1)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (module_floordiv_2): FloorDivide()
            (module_mul): Multiply()
            (module_matmul): MatMul()
            (module_add): Add()
            (module_matmul_1): MatMul()
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
     

Evaluating Accuracy

In [14]:
model

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Sequential(
    (0): BasicLayer(
      dim=96, input_resolution=(56, 56), depth=2
      (blocks): ModuleList(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (a

In [15]:
accuracy = eval_callback(prepared_model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  7.40it/s]

Top-1 Accuracy: 0.9300
Top-5 Accuracy: 0.9900
0.93


In [16]:
accuracy = eval_callback(sim.model)
print(accuracy)

NameError: name 'sim' is not defined

In [17]:
from aimet_common.defs import QuantScheme
from aimet_torch.quantsim import QuantizationSimModel

dummy_input = torch.rand(1, 3, 224, 224).to(device)    # Shape for each ImageNet sample is (3 channels) x (224 height) x (224 width)


sim = QuantizationSimModel(model= prepared_model,
                           quant_scheme=QuantScheme.post_training_tf_enhanced,
                           dummy_input=dummy_input,
                           default_output_bw=8,
                           default_param_bw=8)

2025-03-02 23:10:05,876 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-03-02 23:10:05,997 - Quant - INFO - No config file provided, defaulting to config file at /home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/aimet_common/quantsim_config/default_config.json
2025-03-02 23:10:06,021 - Quant - INFO - Unsupported op type Squeeze
2025-03-02 23:10:06,021 - Quant - INFO - Unsupported op type Mean
2025-03-02 23:10:06,052 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default
2025-03-02 23:10:06,059 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-03-02 23:10:06,060 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-03-02 23:10:06,061 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
202

In [ ]:
print(sim.model)

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [52]:
def pass_calibration_data(model, dataloader, device):
    batch_size = dataloader.batch_size  # Ensure batch_size is correct

    model.eval()
    samples = 1000  

    batch_cntr = 0
    with torch.no_grad():
        for input_data in dataloader:  
            inputs_batch = input_data.to(device)  # Move input to device
            model(inputs_batch)  # Forward pass

            batch_cntr += 1
            if (batch_cntr * batch_size) > samples:
                break


In [19]:
sim.compute_encodings(forward_pass_callback=lambda model: pass_calibration_data(model, dataloader, device))


In [20]:
accuracy = eval_callback(sim.model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.63it/s]

Top-1 Accuracy: 0.8300
Top-5 Accuracy: 0.9800
0.83


To perform quantization aware training (QAT), training the model for a few more epochs 15.

In [31]:
dataset,dataloader =load_labbeled_data(20,20,32)

Dataset size 400 DataLoader 13


In [63]:
import torch
from torch import nn, optim

def train_loop( model: nn.Module, criterion: torch.nn.modules.loss, optimizer: torch.optim,
                    max_iterations: int, current_epoch: int, max_epochs: int,
                    debug_steps: int = 1000 , dataloader : DataLoader = dataloader):

        
        # switch to training mode
        model = model.to(device)
        model.train()

        avg_loss = 0.0

        for i, ( images, target) in tqdm(enumerate(dataloader), total=max_iterations):
            if i == max_iterations:
                break

            images = images.to(device)
            target = target.to(device)

            # compute model output
            output = model(images)
            loss = criterion(output, target)
            avg_loss += loss.item()

            # compute gradient and do SGD step
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % debug_steps == 0:
                eval_accuracy = eval_callback(model)
                print(f"At the end of Epoch #{current_epoch}/{max_epochs}: "
      f"Global Avg Loss={avg_loss / max_iterations:.6f}, Eval Accuracy={eval_accuracy:.4f}")

                # switch to training mode after evaluation
                model.train()

        eval_accuracy = eval_callback(model)
        print("eval : ", eval_accuracy)
        print(f'At the end of Epoch #{current_epoch}/{max_epochs}: '
      f'Global Avg Loss={avg_loss / max_iterations:.6f}, Eval Accuracy={eval_accuracy:.4f}')


        

def train( model: nn.Module, max_epochs: int = 20, learning_rate: int = 0.1,
              weight_decay: float = 1e-4, decay_rate: float = 0.1, learning_rate_schedule: list = None,
              debug_steps: int = 1000, dataloader : DataLoader = dataloader):
        
        max_iterations = len(dataloader)

        loss = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)

        if learning_rate_schedule:
            learning_rate_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, learning_rate_schedule,
                                                                     gamma=decay_rate)

        for current_epoch in range(max_epochs):
            train_loop(model, loss, optimizer, max_iterations, current_epoch + 1, max_epochs, debug_steps,
                             )

            if learning_rate_schedule:
                learning_rate_scheduler.step()

In [64]:
# ImageNetDataPipeline.finetune(sim.model, epochs=1, learning_rate=5e-7, learning_rate_schedule=[5, 10], use_cuda=use_cuda)

train(model=sim.model ,learning_rate=5e-7 , learning_rate_schedule=[5, 10] , max_epochs= 15 , dataloader=dataloader)

100%|██████████| 13/13 [00:04<00:00,  2.72it/s]


Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.36it/s]

Top-1 Accuracy: 0.8500
Top-5 Accuracy: 0.9800
eval :  0.85
At the end of Epoch #1/15: Global Avg Loss=0.493862, Eval Accuracy=0.8500



100%|██████████| 13/13 [00:04<00:00,  3.12it/s]


...

Top-1 Accuracy: 0.8600
Top-5 Accuracy: 0.9900
eval :  0.86
At the end of Epoch #14/15: Global Avg Loss=0.495064, Eval Accuracy=0.8600



100%|██████████| 13/13 [00:04<00:00,  3.10it/s]


Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.33it/s]

Top-1 Accuracy: 0.8600
Top-5 Accuracy: 0.9900
eval :  0.86
At the end of Epoch #15/15: Global Avg Loss=0.508750, Eval Accuracy=0.8600


In [53]:
sim.model.to('cuda:0')

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [37]:

accuracy = eval_callback(sim.model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.06it/s]

Top-1 Accuracy: 0.8600
Top-5 Accuracy: 0.9800
0.86


In [65]:
print("Model Device:", next(sim.model.parameters()).device)  # Prints cuda:0 or cpu
print("Dummy Input Device:", dummy_input.device)  # Prints cuda:0 or cpu
print(device)

Model Device: cuda:0
Dummy Input Device: cuda:0
cuda


In [70]:
print(model.training)  # True if in training mode, False if in eval mode


False


In [72]:
print(dummy_input.shape)

torch.Size([1, 3, 224, 224])


In [73]:
import os
os.makedirs('/media/bmw/harisharajan_r_r/swin/artifacts/qat', exist_ok=True)
sim.export(path='/media/bmw/harisharajan_r_r/swin/artifacts/qat', filename_prefix='swin_after_qat',
 dummy_input=torch.randn(1, 3, 224, 224))

2025-03-02 23:42:17,491 - Utils - INFO - successfully created onnx model with 445/2102 node names updated
2025-03-02 23:42:17,978 - Quant - INFO - Layers excluded from quantization: []
2025-03-02 23:42:17,982 - Quant - WARNING - Quantsim export will stop exporting encodings for saving and loading in a future AIMET release.
To export encodings for saving and loading, use QuantizationSimModel's save_encodings_to_json() utility instead.


Min and Max Value of Encodings 